# Notebook 14: Integración CI/CD

**Duración**: 30 minutos | **Nivel**: Avanzado

## Introducción

Integra Great Expectations en pipelines de CI/CD para validación continua.

### Objetivos:
1. Crear workflow de GitHub Actions
2. Integrar en pipelines de datos
3. Configurar alertas
4. Mejores prácticas de producción

## GitHub Actions Workflow

In [ ]:
# Crear workflow de GitHub Actions
workflow_content = '''name: Data Quality Validation

on:
  push:
    paths:
      - 'data/**'
  pull_request:
    paths:
      - 'data/**'
  schedule:
    - cron: '0 */6 * * *'  # Cada 6 horas

jobs:
  validate:
    runs-on: ubuntu-latest
    
    steps:
    - uses: actions/checkout@v3
    
    - name: Set up Python
      uses: actions/setup-python@v4
      with:
        python-version: '3.10'
    
    - name: Install dependencies
      run: |
        pip install great-expectations pandas
    
    - name: Run validations
      run: |
        python validar.py data/ventas_sucias.csv
    
    - name: Upload Data Docs
      if: always()
      uses: actions/upload-artifact@v3
      with:
        name: data-docs
        path: gx_project/uncommitted/data_docs/
    
    - name: Notify on failure
      if: failure()
      run: |
        echo "Data quality validation failed!"
        # Aquí puedes agregar notificación por Slack, email, etc.
'''

import os
os.makedirs("../.github/workflows", exist_ok=True)
with open("../.github/workflows/data_quality.yml", "w") as f:
    f.write(workflow_content)

print(" GitHub Actions workflow creado")

## Integración con Airflow

In [ ]:
# Ejemplo de DAG de Airflow
airflow_dag = '''from airflow import DAG
from airflow.operators.python import PythonOperator
from datetime import datetime, timedelta
import great_expectations as gx
import pandas as pd

def validar_calidad_datos(**context):
    """Tarea de validación de calidad"""
    # Cargar contexto GX
    gx_context = gx.get_context(mode="file", project_root_dir="./gx_project")
    
    # Cargar datos
    df = pd.read_csv("/path/to/data.csv")
    
    # Ejecutar validación
    datasource = gx_context.data_sources.get("ventas_prod")
    asset = datasource.get_asset("ventas")
    batch_def = asset.get_batch_definition("batch_completo")
    suite = gx_context.suites.get("suite_checkpoint")
    
    val_def = gx_context.validation_definitions.get("validacion_checkpoint")
    resultado = val_def.run(batch_parameters={"dataframe": df})
    
    # Generar Data Docs
    gx_context.build_data_docs()
    
    # Fallar si validación no pasa
    if not resultado.success:
        raise ValueError("Data quality validation failed!")
    
    return "Validation passed"

default_args = {
    'owner': 'data-team',
    'depends_on_past': False,
    'start_date': datetime(2026, 3, 1),
    'email_on_failure': True,
    'email_on_retry': False,
    'retries': 1,
    'retry_delay': timedelta(minutes=5),
}

dag = DAG(
    'data_quality_validation',
    default_args=default_args,
    description='Validación de calidad de datos',
    schedule_interval='@daily',
    catchup=False,
)

validar_task = PythonOperator(
    task_id='validar_calidad',
    python_callable=validar_calidad_datos,
    dag=dag,
)
'''

with open("../airflow_dag_example.py", "w") as f:
    f.write(airflow_dag)

print(" Ejemplo de DAG de Airflow creado")

## Mejores Prácticas de Producción

In [ ]:
best_practices = '''
# Mejores Prácticas para Great Expectations en Producción

## 1. Versionamiento
- Versiona tus expectation suites en Git
- Usa tags para releases
- Documenta cambios en CHANGELOG

## 2. Organización
- Separa suites por criticidad (crítico, advertencia, info)
- Agrupa por dominio de negocio
- Usa nombres descriptivos

## 3. Monitoreo
- Configura alertas para validaciones fallidas
- Monitorea tendencias de calidad en el tiempo
- Crea dashboards de métricas de calidad

## 4. Performance
- Usa sampling para datasets grandes
- Valida en SQL cuando sea posible
- Paraleliza validaciones independientes

## 5. Documentación
- Documenta el propósito de cada expectativa
- Incluye metadata de negocio
- Mantén Data Docs actualizados

## 6. Testing
- Testea tus expectativas con datos de ejemplo
- Valida que las expectativas detectan problemas
- Usa CI/CD para validar cambios

## 7. Mantenimiento
- Revisa expectativas regularmente
- Actualiza umbrales basado en datos históricos
- Elimina expectativas obsoletas
'''

with open("../BEST_PRACTICES.md", "w") as f:
    f.write(best_practices)

print(" Guía de mejores prácticas creada")

##  Ejercicio

Crea un script que:
1. Ejecute validaciones
2. Genere reporte JSON
3. Envíe métricas a sistema de monitoreo

In [ ]:
# TU CÓDIGO AQUÍ
pass

##  Resumen

1.  GitHub Actions para CI/CD
2.  Integración con Airflow
3.  Mejores prácticas de producción
4.  Monitoreo y alertas

**Próximo**: Notebook 15 - Proyecto Integrador 